# 👻 Objets sans galaxie hôte — `hostless_candidate` (Fink/LSST)

Ce notebook est dédié au tag **`hostless_candidate`** qui sélectionne les alertes
pour lesquelles **aucune galaxie hôte n'est détectée** selon l'algorithme ELEPHANT
([arXiv:2404.18165](https://arxiv.org/abs/2404.18165)).

Ces objets sont scientifiquement intéressants car ils peuvent être :
- Des supernovae dans des galaxies naines très peu lumineuses
- Des transitoires dans des régions de faible densité (halos galactiques)
- Des événements extragalactiques lointains sans contrepartie visible

Ce notebook :
1. Compare la **distribution spatiale** hostless vs `sn_near_galaxy_candidate`
2. Compare les **propriétés photométriques** (flux, magnitude, couleur)
3. Affiche les **courbes de lumière** et **cutouts** (absence de source étendue dans le Template)
4. Compare les **scores Fink** entre les deux populations

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources` · `/api/v1/cutouts` · `/api/v1/schema`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_HOSTLESS   = 100        # Alertes hostless_candidate
N_COMPARE    = 100        # Alertes sn_near_galaxy_candidate (pour comparaison)
STARTDATE    = None
STOPDATE     = None
BASE_URL     = "https://api.lsst.fink-portal.org"

# Tag de comparaison
TAG_HOSTLESS = "hostless_candidate"
TAG_COMPARE  = "sn_near_galaxy_candidate"

# Affichage des figures de synthèse (LC + cutouts)
N_OBJECTS_DISPLAY = 5     # Nb max d'objets à afficher en détail
SHOW_CUTOUTS      = True
CUTOUT_CMAP       = 'hot'
CUTOUT_FORMAT     = 'PNG'
CUTOUT_KINDS      = ['Science', 'Template', 'Difference']

SAVE_FIGS  = True
OUTPUT_DIR = "hostless_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
from io import BytesIO
from PIL import Image
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from IPython.display import display as ipy_display, HTML
from tqdm.notebook import tqdm
from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size'         : 11,
    'axes.titlesize'    : 16,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 16,
    'xtick.labelsize'   : 16,
    'ytick.labelsize'   : 16,
    'legend.fontsize'   : 16,
    'figure.titlesize'  : 13,
    'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE',
    'g': '#0077BB',
    'r': '#33AA77',
    'i': '#DDAA33',
    'z': '#BB5500',
    'y': '#AA0000',
}

# Couleurs des deux populations
COLOR_HOSTLESS = '#EE6677'
COLOR_COMPARE  = '#4477AA'

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')
ZP_AB     = 2.5 * np.log10(3631e9)

def mjd_to_datetime(s):
    return MJD_EPOCH + pd.to_timedelta(pd.to_numeric(s, errors='coerce'), unit='D')

def mjd_to_datestr(m):
    try: return (MJD_EPOCH + pd.to_timedelta(float(m), unit='D')).strftime('%Y-%m-%d')
    except: return str(m)

def flux_to_mag(flux, flux_e):
    flux, flux_e = np.asarray(flux, float), np.asarray(flux_e, float)
    v = flux > 0
    mag = np.full(flux.shape, np.nan)
    me  = np.full(flux.shape, np.nan)
    mag[v] = -2.5 * np.log10(flux[v]) + ZP_AB
    me[v]  = (2.5 / np.log(10)) * np.abs(flux_e[v] / flux[v])
    return mag, me

def radec_to_mollweide(ra, dec):
    return -np.deg2rad(np.asarray(ra) - 180.), np.deg2rad(np.asarray(dec))

def split_curve(x, y):
    breaks = np.where(np.abs(np.diff(x)) > np.pi * 0.8)[0] + 1
    return np.split(x, breaks), np.split(y, breaks)

print("Imports OK ✓")

## 3. Récupération des deux populations

In [ ]:
# Schéma — colonnes clf
# L'API retourne un dict à deux sections :
#   "LSST original fields (r:)" : champs originaux LSST
#   "Fink added values (f:)"    : valeurs ajoutées par Fink
resp = requests.post(f"{BASE_URL}/api/v1/schema",
                     json={"endpoint": "/api/v1/sources"}, timeout=60)
_schema = resp.json()
_fink_section = next(
    (v for k, v in _schema.items() if 'fink' in k.lower() or k.startswith('f:')),
    {}
)
clf_cols = [f"f:{k}" for k in _fink_section if k.startswith('clf')]
print(f"Colonnes clf trouvées ({len(clf_cols)}) : {clf_cols}")


def fetch_population(tag, n, startdate=None, stopdate=None):
    """Récupère alertes d'un tag + sources enrichies par lots."""
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate: payload["startdate"] = startdate
    if stopdate:  payload["stopdate"]  = stopdate
    r = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    r.raise_for_status()
    if not r.json(): return pd.DataFrame()
    df_tags = pd.DataFrame(r.json())
    df_tags.columns = [c.replace('r:','').replace('f:','') for c in df_tags.columns]
    id_col = next(c for c in df_tags.columns if 'diaObjectId' in c)
    oids   = df_tags[id_col].astype(str).unique().tolist()

    base = "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,r:band,r:psfFlux,r:psfFluxErr,r:ra,r:dec"
    extra = ','.join(clf_cols[:4]) if clf_cols else ''
    cols  = base + (',' + extra if extra else '')

    parts = []
    for chunk in [oids[i:i+20] for i in range(0, len(oids), 20)]:
        try:
            r2 = requests.post(f"{BASE_URL}/api/v1/sources",
                               json={"diaObjectId": ','.join(chunk),
                                     "columns": cols, "output-format": "json"},
                               timeout=120)
            r2.raise_for_status()
            if r2.json():
                df = pd.DataFrame(r2.json())
                df.columns = [c.replace('r:','').replace('f:','') for c in df.columns]
                parts.append(df)
        except Exception as e:
            print(f"  ✗ lot : {e}")

    df_all = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    for col in ['midpointMjdTai','psfFlux','psfFluxErr','ra','dec']:
        if col in df_all.columns:
            df_all[col] = pd.to_numeric(df_all[col], errors='coerce')
    df_all['tag'] = tag
    return df_all


print(f"Récupération '{TAG_HOSTLESS}'...")
df_host = fetch_population(TAG_HOSTLESS, N_HOSTLESS, STARTDATE, STOPDATE)
print(f"  ✓ {len(df_host)} sources, {df_host['diaObjectId'].nunique()} objets")

print(f"\nRécupération '{TAG_COMPARE}'...")
df_comp = fetch_population(TAG_COMPARE,  N_COMPARE,  STARTDATE, STOPDATE)
print(f"  ✓ {len(df_comp)} sources, {df_comp['diaObjectId'].nunique()} objets")


## 4. Comparaison spatiale — carte Mollweide

In [ ]:
# Plan galactique
l = np.linspace(0, 360, 1000)
gc = SkyCoord(l=l*u.deg, b=np.zeros(1000)*u.deg, frame='galactic').icrs
idx_g = np.argsort(gc.ra.deg)
gal_x, gal_y = radec_to_mollweide(gc.ra.deg[idx_g], gc.dec.deg[idx_g])

def pos_per_obj(df):
    return df.groupby('diaObjectId').agg(ra=('ra','median'), dec=('dec','median')).reset_index()

pos_h = pos_per_obj(df_host)
pos_c = pos_per_obj(df_comp)

fig = plt.figure(figsize=(16, 8), layout='constrained')
ax  = fig.add_subplot(111, projection='mollweide')
fig.suptitle(f"Distribution spatiale — hostless vs sn_near_galaxy")
ax.grid(True, color='gray', alpha=0.3, linewidth=0.5, linestyle='--')

for sx, sy in zip(*split_curve(gal_x, gal_y)):
    ax.plot(sx, sy, color='goldenrod', lw=1.2)

for df_p, color, label in [
    (pos_c, COLOR_COMPARE,  f"{TAG_COMPARE} ({len(pos_c)})"),
    (pos_h, COLOR_HOSTLESS, f"{TAG_HOSTLESS} ({len(pos_h)})"),
]:
    sub = df_p.dropna(subset=['ra','dec'])
    px, py = radec_to_mollweide(sub['ra'].values, sub['dec'].values)
    ax.scatter(px, py, s=15, color=color, alpha=0.75, zorder=5,
               edgecolors='white', linewidths=0.3, label=label)

ax.legend(loc='lower left', fontsize=9, framealpha=0.75)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/hostless_skymap.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/hostless_skymap.png", bbox_inches='tight', dpi=150)
    print("✓ Sauvegardé")
plt.show()

## 5. Comparaison photométrique — flux, magnitude, couleur

Distributions côte à côte pour les deux populations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout='constrained')
fig.suptitle("Comparaison photométrique — hostless vs sn_near_galaxy")

kw_h = dict(color=COLOR_HOSTLESS, alpha=0.6, density=True,
            edgecolor='white', linewidth=0.3, label=TAG_HOSTLESS.replace('_candidate',''))
kw_c = dict(color=COLOR_COMPARE,  alpha=0.6, density=True,
            edgecolor='white', linewidth=0.3, label=TAG_COMPARE.replace('_candidate',''))

# Distribution flux (log)
for df_, kw in [(df_host, kw_h), (df_comp, kw_c)]:
    f = df_['psfFlux'].dropna()
    f = f[f > 0]
    if not f.empty:
        axes[0].hist(np.log10(f), bins=30, **kw)
axes[0].set_xlabel('log₁₀ Flux PSF (nJy)')
axes[0].set_ylabel('Densité')
axes[0].set_title('Distribution flux')
axes[0].legend(fontsize=8)
axes[0].grid(True, axis='y', alpha=0.25, linewidth=0.5)

# Distribution magnitude
for df_, kw in [(df_host, kw_h), (df_comp, kw_c)]:
    f = df_['psfFlux'].dropna()
    mag = -2.5 * np.log10(f[f > 0]) + ZP_AB
    if not mag.empty:
        axes[1].hist(mag, bins=30, range=(19, 28), **kw)
axes[1].set_xlabel('Magnitude AB')
axes[1].set_ylabel('Densité')
axes[1].set_title('Distribution magnitude')
axes[1].legend(fontsize=8)
axes[1].grid(True, axis='y', alpha=0.25, linewidth=0.5)

# Distribution déclinaison
for df_, kw in [(df_host, kw_h), (df_comp, kw_c)]:
    d = df_['dec'].dropna()
    if not d.empty:
        axes[2].hist(d, bins=25, range=(-70, 10), **kw)
axes[2].axvline(-62, color='gray', lw=1.0, ls='--', alpha=0.7, label='~limite LSST')
axes[2].set_xlabel('Déclinaison δ (deg)')
axes[2].set_ylabel('Densité')
axes[2].set_title('Distribution Dec')
axes[2].legend(fontsize=8)
axes[2].grid(True, axis='y', alpha=0.25, linewidth=0.5)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/hostless_phot_comp.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/hostless_phot_comp.png", bbox_inches='tight', dpi=150)
    print("✓ Sauvegardé")
plt.show()

## 6. Comparaison des scores Fink

In [ ]:
clf_present = [c.replace('f:','') for c in clf_cols
               if c.replace('f:','') in df_host.columns
               and c.replace('f:','') in df_comp.columns]

if not clf_present:
    print("⚠️  Aucun score clf commun disponible.")
else:
    n_clf = len(clf_present)
    fig, axes = plt.subplots(1, n_clf, figsize=(5*n_clf, 4), layout='constrained')
    if n_clf == 1: axes = [axes]
    fig.suptitle("Scores de classification — hostless vs sn_near_galaxy")

    for ax, col in zip(axes, clf_present):
        short = col.replace('clf.','')
        for df_, color, label in [
            (df_comp, COLOR_COMPARE,  TAG_COMPARE.replace('_candidate','')),
            (df_host, COLOR_HOSTLESS, TAG_HOSTLESS.replace('_candidate','')),
        ]:
            vals = pd.to_numeric(df_[col], errors='coerce').dropna()
            if not vals.empty:
                ax.hist(vals, bins=25, range=(0,1),
                        color=color, alpha=0.6, density=True,
                        edgecolor='white', linewidth=0.3, label=f"{label} (n={len(vals)})")
        ax.axvline(0.5, color='black', lw=1.0, ls='--', alpha=0.6)
        ax.set_title(short, fontsize=10)
        ax.set_xlabel('Score')
        ax.set_ylabel('Densité')
        ax.set_xlim(0, 1)
        ax.legend(fontsize=8)
        ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/hostless_scores.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/hostless_scores.png", bbox_inches='tight', dpi=150)
        print("✓ Sauvegardé")
    plt.show()

## 7. Figures de synthèse — courbes de lumière + cutouts

Le Template doit montrer l'**absence** de source étendue (galaxie hôte) pour les hostless.

In [ ]:
# Sélectionner les N_OBJECTS_DISPLAY premiers objets hostless (par flux max décroissant)
flux_max_per_obj = df_host.groupby('diaObjectId')['psfFlux'].max().sort_values(ascending=False)
top_oids = flux_max_per_obj.head(N_OBJECTS_DISPLAY).index.tolist()

for oid in top_oids:
    lc = df_host[df_host['diaObjectId'].astype(str)==str(oid)].copy()
    lc = lc.sort_values('midpointMjdTai')
    lc['date'] = mjd_to_datetime(lc['midpointMjdTai'])

    lc_s       = lc.dropna(subset=['midpointMjdTai']).sort_values('midpointMjdTai', ascending=False)
    last_row   = lc_s.iloc[0]
    last_src   = last_row.get('diaSourceId', None)
    last_band  = last_row.get('band', 'N/A')
    last_date  = mjd_to_datestr(last_row.get('midpointMjdTai'))

    score_str = ''
    if clf_present:
        s = pd.to_numeric(lc[clf_present[0]], errors='coerce').median()
        if not np.isnan(s):
            score_str = f"   {clf_present[0].replace('clf.','')} : {s:.3f}"

    title = (f"diaObjectId : {oid}   |   "
             f"hostless_candidate   "
             f"diaSourceId : {last_src}   "
             f"filtre : {last_band}   date : {last_date}"
             f"{score_str}")

    FIG_W    = 16.0
    CUT_SIZE = FIG_W * 0.90 * 0.94 / 3
    LC_H     = CUT_SIZE * 0.90

    fig = plt.figure(figsize=(FIG_W, LC_H + CUT_SIZE + 1), layout='constrained')
    fig.suptitle(title, fontsize=10, fontweight='bold')

    gs     = GridSpec(2, 1, figure=fig, height_ratios=[LC_H, CUT_SIZE])
    gs_lc  = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[0], wspace=0.07)
    ax_f   = fig.add_subplot(gs_lc[0])
    ax_m   = fig.add_subplot(gs_lc[1])
    gs_bot = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[1],
                                     width_ratios=[0.94, 0.06], wspace=0.02)
    gs_cut = GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_bot[0], wspace=0.03)
    axes_cut = [fig.add_subplot(gs_cut[i]) for i in range(3)]
    ax_cbar  = fig.add_subplot(gs_bot[1])
    ax_cbar.set_visible(False)

    # Courbes de lumière
    bands_ok = sorted(lc['band'].dropna().unique())
    for band in bands_ok:
        sub   = lc[lc['band']==band].dropna(subset=['psfFlux','psfFluxErr'])
        color = FILTER_COLORS.get(band,'#888')
        ax_f.errorbar(sub['date'], sub['psfFlux'], yerr=sub['psfFluxErr'],
                      fmt='o-', color=color, label=f"${band}$",
                      ms=4, capsize=3, lw=1.0, alpha=0.85)
        mag_v, me_v = flux_to_mag(sub['psfFlux'].values, sub['psfFluxErr'].values)
        det = ~np.isnan(mag_v)
        if det.any():
            ax_m.errorbar(sub['date'].values[det], mag_v[det], yerr=me_v[det],
                          fmt='o-', color=color, label=f"${band}$",
                          ms=4, capsize=3, lw=1.0, alpha=0.85)

    ax_f.axhline(0, color='gray', lw=0.6, ls='--', alpha=0.5)
    ax_f.set_ylabel('Flux PSF (nJy)')
    ax_f.legend(ncol=len(bands_ok), loc='upper left', handlelength=1)
    for ax_ in [ax_f, ax_m]:
        ax_.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.setp(ax_.xaxis.get_majorticklabels(), rotation=30, ha='right')
        ax_.grid(True, alpha=0.2, linewidth=0.5)
    ax_m.invert_yaxis()
    ax_m.set_ylabel('Magnitude AB')
    ax_m.legend(ncol=len(bands_ok), loc='upper left', handlelength=1)

    # Cutouts — le Template doit être vide (pas de source étendue)
    if SHOW_CUTOUTS and last_src is not None:
        arrays, pairs = [], []
        for ax_c, kind in zip(axes_cut, CUTOUT_KINDS):
            try:
                r3 = requests.get(f"{BASE_URL}/api/v1/cutouts",
                                  params={"diaSourceId": str(last_src),
                                          "kind": kind, "output-format": CUTOUT_FORMAT},
                                  timeout=60)
                r3.raise_for_status()
                arr = np.array(Image.open(BytesIO(r3.content))).astype(float)
                if arr.ndim == 3: arr = arr.mean(axis=2)
                arrays.append(arr)
                im = ax_c.imshow(arr, cmap=CUTOUT_CMAP, origin='upper', aspect='auto')
                ax_c.axis('off')
                # Annotation spéciale pour le Template
                note = " ← pas de hôte" if kind == 'Template' else ""
                ax_c.set_title(f"{kind}{note}\n{last_band}  {last_date}", fontsize=9)
                pairs.append((ax_c, im, arr))
            except Exception as e:
                ax_c.axis('off')
                ax_c.text(0.5, 0.5, f"{kind}\nerreur", ha='center', va='center',
                          transform=ax_c.transAxes, fontsize=8, color='red')
        if pairs:
            norm_c = plt.Normalize(vmin=min(a.min() for _,_,a in pairs),
                                   vmax=max(a.max() for _,_,a in pairs))
            fig.canvas.draw()
            pos_d = axes_cut[-1].get_position()
            pos_s = ax_cbar.get_position()
            cax   = fig.add_axes([pos_s.x0+pos_s.width*0.10, pos_d.y0,
                                   pos_s.width*0.35, pos_d.height])
            sm = plt.cm.ScalarMappable(cmap=CUTOUT_CMAP, norm=norm_c)
            sm.set_array([])
            fig.colorbar(sm, cax=cax, label='ADU')
    else:
        for ax_c in axes_cut: ax_c.axis('off')

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/hostless_{oid}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/hostless_{oid}.png", bbox_inches='tight', dpi=150)

    plt.show()
    ipy_display(HTML(
        f'<div style="font-size:12px;margin:2px 0 14px 4px;">'
        f'🔗 <b>Portail Fink/LSST</b> : '
        f'<a href="https://lsst.fink-portal.org/{oid}" target="_blank">'
        f'https://lsst.fink-portal.org/{oid}</a></div>'
    ))
    print()

print("\n✅ Affichage terminé.")

## 8. Tableau récapitulatif — objets hostless

In [ ]:
agg = {'ra': 'median', 'dec': 'median', 'psfFlux': 'max',
       'midpointMjdTai': 'max',
       'band': lambda x: ', '.join(sorted(x.dropna().unique()))}
for c in clf_present:
    agg[c] = 'median'

df_sum = df_host.groupby('diaObjectId').agg(agg).reset_index()
df_sum['last_date'] = df_sum['midpointMjdTai'].apply(mjd_to_datestr)
df_sum['mag_pic']   = df_sum['psfFlux'].apply(
    lambda f: round(-2.5*np.log10(f)+ZP_AB, 2) if f > 0 else np.nan)
df_sum['ra']        = df_sum['ra'].round(5)
df_sum['dec']       = df_sum['dec'].round(5)
df_sum['psfFlux']   = df_sum['psfFlux'].round(1)
for c in clf_present:
    if c in df_sum.columns:
        df_sum[c] = df_sum[c].round(3)

if clf_present:
    df_sum = df_sum.sort_values(clf_present[0], ascending=False)

df_sum['Portail Fink'] = df_sum['diaObjectId'].apply(
    lambda o: f'<a href="https://lsst.fink-portal.org/{o}" target="_blank">🔗 {o}</a>')

keep = ['diaObjectId','ra','dec','last_date','psfFlux','mag_pic','band']
keep += [c for c in clf_present if c in df_sum.columns]
keep += ['Portail Fink']
keep = [c for c in keep if c in df_sum.columns]

html = df_sum[keep].to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse:collapse; font-size:12px; width:100%; }
  .fink-table th { background:#f0f0f0; padding:6px 10px;
                   border-bottom:2px solid #ccc; text-align:left; }
  .fink-table td { padding:4px 10px; border-bottom:1px solid #eee;
                   text-align:left; white-space:nowrap; }
  .fink-table tr:hover td { background:#fafafa; }
</style>
"""
ipy_display(HTML(style + html))